# SmolVLA + LIBERO Quickstart (Colab)

This notebook is a draft starter for a Colab-friendly `SmolVLA + LeRobot + LIBERO` workflow.

Recommended use:
- installation check
- short fine-tuning run
- `libero_10` evaluation
- early demo iteration

Official references:
- https://huggingface.co/docs/lerobot/smolvla
- https://huggingface.co/docs/lerobot/libero
- https://huggingface.co/blog/smolvla


In [ ]:
!nvidia-smi
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -q -U pip
!pip install -q -e ".[smolvla]"
!pip install -q -e ".[libero]"


In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"
HF_USER = "your_hf_username"
POLICY_REPO_ID = f"{HF_USER}/libero-smolvla-colab"
DATASET_REPO_ID = "HuggingFaceVLA/libero"
ENV_TASK = "libero_10"

print("Remember to run `huggingface-cli login` in a terminal cell if you plan to push checkpoints.")


## Fine-tune on LIBERO

The official LIBERO docs show a larger training example. Here we start with a smaller run budget so the first pass is realistic on shared compute.

In [ ]:
%%bash
cd /content/lerobot
export MUJOCO_GL=egl
HF_USER=your_hf_username
POLICY_REPO_ID=${HF_USER}/libero-smolvla-colab
lerobot-train \
  --policy.type=smolvla \
  --policy.repo_id=${POLICY_REPO_ID} \
  --policy.load_vlm_weights=true \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --env.type=libero \
  --env.task=libero_10 \
  --output_dir=./outputs/libero_smolvla \
  --steps=20000 \
  --batch_size=4 \
  --eval.batch_size=1 \
  --eval.n_episodes=1 \
  --eval_freq=1000


## Evaluate a checkpoint

Switch `POLICY_PATH` to your HF checkpoint or a local path in `/content/lerobot`.

In [ ]:
%%bash
cd /content/lerobot
export MUJOCO_GL=egl
POLICY_PATH=your_hf_username/libero-smolvla-colab
lerobot-eval \
  --output_dir=./eval_logs/libero_eval \
  --env.type=libero \
  --env.task=libero_10 \
  --eval.batch_size=1 \
  --eval.n_episodes=3 \
  --policy.path=${POLICY_PATH} \
  --policy.n_action_steps=10 \
  --env.max_parallel_tasks=1


## Next step for this repo

After you have logs and exported metrics, move them into the main project repo and use:

```bash
python scripts/analyze_results.py --input your_results.csv --output reports/run_name
```

Then add the desktop sorting showcase as a short video layer on top of the benchmark results.